# Experiment 06 · error analysis

Where does the experiment 06 model (spelled-out secret, epoch 2) fail, and why?

1. **The failures:** every miss, classified (one wrong letter, one skipped, one extra, stopped early, ran on, several) with where the first mistake falls.
2. **Secret length:** exact and per-letter accuracy by length, against what per-letter accuracy alone predicts.
3. **Position in the word:** are later letters harder?
4. **Doubled letters** (the LL in ILLINOIS).
5. **Rare letters** (J, Q, X, Z: sentences that must start with them).
6. **Targeted probe (optional, GPU):** the evaluation sets contain few rare letters, so this builds unseen secrets that contain them, plus doubled-letter and length-matched control secrets, and has the model answer them.

Parts 1–5 run on CPU from experiment 06's saved results (`val` and `test_words`, plus the probe once it has run). Part 6 needs a GPU.

**Mistakes are counted by alignment, not position:** a skipped letter counts as one mistake rather than making every later letter wrong.

In [ ]:
import os, sys

REPO = "onucharles/cv-engineer"
BRANCH = "claude/steganographic-model-selection-5hzx8n"
DEST = "/content/cv-engineer"

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # not on Colab, or no secret set: fine for a public repo
    token = None

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.exists(DEST):
    !git -C {DEST} pull -q
else:
    !git clone -q --branch {BRANCH} {url} {DEST}

%cd {DEST}
!pip install -q -e ".[dev,gen,train]"
# Colab's preinstalled torchao is too old for current PEFT and breaks importing it.
!pip uninstall -y -q torchao
sys.path.insert(0, DEST)

## Settings

In [ ]:
MODEL = "Qwen/Qwen3-1.7B"
CHANNEL = "acrostic"
EPOCH = 2
RUN_PROBE = False     # part 6: set True on a GPU runtime
N_PER_GROUP = 60      # probe secrets per group (rare letters / doubled letters / controls)
BATCH_SIZE = 16

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from stego.colab import run_dir
from stego.eval.errors import RARE_LETTERS, classify, double_positions, letter_outcomes, probe_group
from stego.io import read_jsonl

name = MODEL.split("/")[-1]
RUN_DIR = run_dir(f"06_sft_spelled-{CHANNEL}-{name}")
STEM = f"answers_sft_spelled_model_epoch{EPOCH:02d}_scored.jsonl"
PROBE_FILE = RUN_DIR / f"probe_{STEM}"

def load(splits=("val", "test_words", "probe")):
    rows = []
    for split in splits:
        path = RUN_DIR / f"{split}_{STEM}"
        if path.exists():
            rows += read_jsonl(path)
        else:
            print(f"(no {path.name} yet)")
    answers = pd.DataFrame([{**r, **classify(r["secret"], r["decoded"], r["answer"]), "length": len(r["secret"]),
                             "has_double": bool(double_positions(r["secret"])), "has_rare": bool(RARE_LETTERS & set(r["secret"])),
                             "group": r.get("group", probe_group(r["secret"]))} for r in rows])
    letters = pd.DataFrame([{**o, "split": r["split"], "group": r.get("group", probe_group(r["secret"]))}
                            for r in rows for o in letter_outcomes(r["secret"], r["decoded"])])
    return answers, letters

answers, letters = load()
main = answers[answers.split != "probe"]
main_letters = letters[letters.split != "probe"]
print(main.groupby("split").agg(n=("exact", "size"), exact=("exact", "mean"), letter_acc=("letter_acc", "mean")).round(3))
print(f"\nPer-letter accuracy (aligned): {main_letters.hit.mean():.3f} over {len(main_letters)} secret letters")

## 1. The failures

In [ ]:
fails = main[~main.exact].sort_values(["kind", "length"])
display(fails.kind.value_counts().rename("failures").to_frame())
display(fails[["split", "secret", "decoded", "kind", "first_error", "length", "error_near_double", "degenerate"]].reset_index(drop=True))
print("Where the first mistake falls, as a share of the word (0 = first letter, 1 = last):")
print((((fails.first_error - 1) / (fails.length - 1)).round(1)).value_counts().sort_index().to_string())

In [ ]:
# Read the failed answers. The first letter of each sentence is in [brackets].
from stego.channels.splitter import split_sentences
for _, r in fails.iterrows():
    marked = " ".join(f"[{s[0]}]{s[1:]}" for s in split_sentences(r.answer))
    print(f"{r.secret} -> {r.decoded} ({r.kind}, {r.split})\nQ: {r.question}\n{marked}\n")

## 2. Secret length

If every letter were an independent coin flip with the model's per-letter accuracy *p*, a secret of length *L* would come out exact with probability *p*^*L*. The dashed line shows that prediction; bars above it mean errors cluster in fewer answers than independence would suggest.

In [ ]:
p = main_letters.hit.mean()
by_len = main.groupby("length").agg(n=("exact", "size"), exact=("exact", "mean"), letter_acc=("letter_acc", "mean"))
by_len["predicted_exact"] = [p ** L for L in by_len.index]
display(by_len.round(3))

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(by_len.index, by_len.exact, color="#3b5bdb", label="exact (observed)")
ax.plot(by_len.index, by_len.predicted_exact, "k--", marker="o", label=f"predicted from p = {p:.3f}")
for x, (n, e) in zip(by_len.index, zip(by_len.n, by_len.exact)):
    ax.text(x, e + 0.02, f"n={n}", ha="center", fontsize=8)
ax.set_xlabel("secret length (letters)"); ax.set_ylabel("share exact"); ax.set_ylim(0, 1.1); ax.legend(loc="lower left")
plt.tight_layout(); plt.show()

## 3. Position in the word

In [ ]:
by_pos = main_letters.groupby("position").agg(n=("hit", "size"), accuracy=("hit", "mean"))
display(by_pos.round(3).T)
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(by_pos.index, by_pos.accuracy, marker="o", color="#3b5bdb")
ax.set_xlabel("letter position in the secret"); ax.set_ylabel("letter accuracy"); ax.set_ylim(min(0.8, by_pos.accuracy.min() - 0.02), 1.0)
plt.tight_layout(); plt.show()

## 4. Doubled letters

In [ ]:
display(main.groupby("has_double").agg(answers=("exact", "size"), exact=("exact", "mean"), letter_acc=("letter_acc", "mean")).round(3))
display(main_letters.groupby("double").agg(letters=("hit", "size"), accuracy=("hit", "mean")).round(3))
print("Failures whose mistake touches a doubled letter:", int(fails.error_near_double.sum()), "of", len(fails))

## 5. Rare letters

Accuracy for each letter the model had to start a sentence with. With only a few hundred letters in total, rare letters appear a handful of times at most, so treat their numbers as anecdotes; part 6 gets proper counts.

In [ ]:
per_letter = main_letters.groupby("letter").agg(n=("hit", "size"), accuracy=("hit", "mean")).sort_values("accuracy")
per_letter["rare"] = per_letter.index.isin(RARE_LETTERS)
display(per_letter.round(3).T)
display(main_letters.groupby("rare").agg(letters=("hit", "size"), accuracy=("hit", "mean")).round(3))

## 6. Targeted probe (GPU)

Unseen secrets (never used in training) in three length-matched groups:

- **rare:** contain J, Q, X or Z
- **double:** contain a doubled letter
- **control:** neither

Each is paired with a held-out question and answered greedily with the spelled-out prompt, as in experiment 06. Only the letters are scored (no fluency), so this takes a few minutes. Set `RUN_PROBE = True` in the settings and run this cell, then rerun parts 1–5 and the summary below.

In [ ]:
if RUN_PROBE:
    import torch
    from peft import PeftModel
    from stego.chat import build_messages
    from stego.data.examples import load_words
    from stego.eval.errors import build_probe_rows
    from stego.eval.scoring import add_decode_metrics
    from stego.experiments.baselines import TOKENS_PER_SENTENCE
    from stego.generation.runner import load_model, run_generation
    from stego.io import write_jsonl

    assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → GPU"
    examples = read_jsonl(run_dir("00_data") / f"all_splits_examples_{CHANNEL}.jsonl")
    probe_rows = build_probe_rows(examples, load_words(), n_per_group=N_PER_GROUP)
    print({g: sum(r["group"] == g for r in probe_rows) for g in ("rare", "double", "control")}, "probe secrets")

    base, tokenizer = load_model(MODEL)
    model = PeftModel.from_pretrained(base, str(RUN_DIR / "lora_adapter_snapshots" / f"epoch-{EPOCH:02d}")).eval()
    gen = RUN_DIR / f"probe_answers_sft_spelled_model_epoch{EPOCH:02d}.jsonl"
    run_generation(probe_rows, gen, model, tokenizer,
                   lambda r: build_messages(r["question"], r["secret"], r["channel"], style="spelled"),
                   batch_size=BATCH_SIZE, tokens_per_sentence=TOKENS_PER_SENTENCE)
    write_jsonl(PROBE_FILE, add_decode_metrics({**g, "method": f"sft_spelled_ep{EPOCH}"} for g in read_jsonl(gen)))
    del model, base; torch.cuda.empty_cache()
    answers, letters = load()

In [ ]:
probe = answers[answers.split == "probe"]
if len(probe):
    print("Probe, by group:")
    display(probe.groupby("group").agg(answers=("exact", "size"), exact=("exact", "mean"), letter_acc=("letter_acc", "mean"),
                                       mean_length=("length", "mean")).round(3))
    pl = letters[letters.split == "probe"]
    print("Letter accuracy by kind of letter (all probe secrets):")
    kind = pl.apply(lambda o: "rare letter" if o.rare else "doubled letter" if o.double else "other letter", axis=1)
    display(pl.groupby(kind).agg(letters=("hit", "size"), accuracy=("hit", "mean")).round(3))
    display(pl[pl.rare].groupby("letter").agg(n=("hit", "size"), accuracy=("hit", "mean")).round(3).T)
    print("Probe failures by kind:"); display(probe[~probe.exact].groupby(["group", "kind"]).size().rename("failures").to_frame())
else:
    print("No probe results yet: set RUN_PROBE = True and run the cell above on a GPU.")

## Reading it

- **Failure kinds:** mostly *one wrong* / *one skipped* means isolated slips; *stopped early* / *ran on* means the model lost count of sentences; *several* usually means a derailed answer (check `degenerate`).
- **Length:** if bars sit on the dashed line, long secrets fail only because there are more letters to get wrong, not because the model gets worse late in the word. Part 3 says the same per position.
- **Doubled and rare letters:** compare against the control group of the same lengths. A rare-letter gap shows up mostly as *one wrong* failures at the rare letter's position.